In [32]:
from pathlib import Path
import pandas as pd

In [33]:
Path("../transactions.parquet").stat().st_size / 1024 / 1024
Path("../transactions_25M.parquet").stat().st_size / 1024 / 1024
Path("../transactions_50M.parquet").stat().st_size / 1024 / 1024

1194.6014966964722

In [ ]:

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.read.parquet("/{{actually_path_to_parquet}}")
print(df.rdd.getNumPartitions())

8


In [24]:
summary = pd.read_csv(Path("../results/sweep_20261003-155739/sweep_summary.csv"))
summary["framework"] = summary["framework"].str.replace(" (local[*])", "", regex=False)

# fastest framework per benchmark at each size
fastest = summary.loc[summary.groupby(["benchmark", "size_m"])["median_s"].idxmin(),
                    ["benchmark", "size_m", "framework", "median_s", "rows_per_sec"]]
# print(fastest.sort_values(["benchmark", "size_m"]))
print(summary.pivot_table(index=["benchmark", "size_m"],
                            columns="framework",
                            values="median_s").round(2))

framework         Pandas  Polars  Spark
benchmark size_m                       
Aggregate 10        4.09    1.56   1.10
          25       10.88    1.50   1.74
          50       20.93    3.44   5.48
Ingest    10        2.98    4.08   1.13
          25       10.38    6.56   6.22
          50       33.16   39.76  15.32
Transform 10        1.02    0.09   0.64
          25        9.38   11.53   7.37
          50       27.00   21.19  14.45


In [25]:
summary = pd.read_csv(Path("../results/sweep_20261003-155739/sweep_summary.csv"))
summary["framework"] = summary["framework"].str.replace(" (local[*])", "", regex=False)

min_times = summary.groupby(["benchmark", "size_m"])["median_s"].transform("min")
summary["vs_best"] = (summary["median_s"] / min_times).round(2)

best_fw = summary.loc[summary.groupby(["benchmark", "size_m"])["median_s"].idxmin(),
                    ["benchmark", "size_m", "framework"]].rename(columns={"framework": "best"})
summary = summary.merge(best_fw, on=["benchmark", "size_m"])

print(summary[["benchmark", "size_m", "framework", "median_s", "vs_best", "best"]]
    .sort_values(["benchmark", "size_m", "framework"]))

    benchmark  size_m framework  median_s  vs_best    best
7   Aggregate      10    Pandas    4.0857     3.73   Spark
6   Aggregate      10    Polars    1.5647     1.43   Spark
8   Aggregate      10     Spark    1.0953     1.00   Spark
16  Aggregate      25    Pandas   10.8841     7.24  Polars
15  Aggregate      25    Polars    1.5037     1.00  Polars
17  Aggregate      25     Spark    1.7357     1.15  Polars
25  Aggregate      50    Pandas   20.9290     6.08  Polars
24  Aggregate      50    Polars    3.4415     1.00  Polars
26  Aggregate      50     Spark    5.4757     1.59  Polars
1      Ingest      10    Pandas    2.9829     2.65   Spark
0      Ingest      10    Polars    4.0829     3.62   Spark
2      Ingest      10     Spark    1.1272     1.00   Spark
10     Ingest      25    Pandas   10.3757     1.67   Spark
9      Ingest      25    Polars    6.5625     1.05   Spark
11     Ingest      25     Spark    6.2228     1.00   Spark
19     Ingest      50    Pandas   33.1572     2.16   Spa